In [1]:
import sys; sys.path.append('..')
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simulator.loader import MarketDataLoader
from simulator.hmm import HybridJumpsHMM
from simulator.single_index import SingleIndexModel
from experts.markowitz import MarkowitzExpert
from torch.utils.data import random_split

# 1. Load config and extract variables cleanly
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

market_ticker = str(config["data"]["market_ticker"])
stock_tickers = list(config["data"]["stock_tickers"])

loader = MarketDataLoader(
    tickers=[market_ticker] + stock_tickers,
    start_date=config["data"]["start_date"], 
    end_date=config["data"]["end_date"]
)
data = loader.fetch_data()

# 2. Simulate standard HMM data
hmm = HybridJumpsHMM(n_states=config["hmm"]["n_states"])
hmm.fit(data[market_ticker])
best_eps, best_lambd = hmm.grid_search(data[market_ticker], max_lag=20, n_paths=10)

sim_model = SingleIndexModel(tickers=stock_tickers)
sim_model.fit(data[stock_tickers], data[market_ticker])
# sim_states = hmm.simulate_states(config["hmm"]["simulation"]["n_steps"], best_eps, best_lambd)
sim_states = hmm.simulate_states(config["hmm"]["simulation"]["n_steps"], epsilon=0.0, lambd=0.0)
sim_stocks = sim_model.simulate(hmm.decode_states(sim_states))

# 3. Get Ground Truth Labels from the Expert
expert = MarkowitzExpert(
    risk_aversion=config["expert"]["risk_aversion"],
    rolling_window=config["expert"]["rolling_window"],
    max_weight=config["expert"]["max_weight"],
    annual_risk_free_rate=config["expert"]["annual_risk_free_rate"]
)
expert_weights_df = expert.generate_labels(sim_stocks)

print(f"Data ready! Total trading days: {len(expert_weights_df)}")

Data ready! Total trading days: 2460


In [2]:
from agents.dataset import PortfolioDataset
from agents.diffusion import MLPDenoiser, TransformerDenoiser, PortDiff
from agents.trainer import PortDiffTrainer
from torch.utils.data import Subset
import torch
import numpy as np

window_size = config["expert"]["rolling_window"]

# 1. Create Dual Datasets
dataset_mlp = PortfolioDataset(
    returns_df=sim_stocks,
    weights_df=expert_weights_df,
    window_size=window_size,
    flatten=True
)

dataset_tf = PortfolioDataset(
    returns_df=sim_stocks,
    weights_df=expert_weights_df,
    window_size=window_size,
    flatten=False
)

# 2. Chronological Split with Purge Gap (Zero Data Leakage)
total_days = len(dataset_mlp)
train_size = int(0.8 * total_days)

train_indices = list(range(0, train_size))
val_start = train_size + window_size
val_indices = list(range(val_start, total_days))

# Split MLP Dataset
train_dataset_mlp = Subset(dataset_mlp, train_indices)
val_dataset_mlp = Subset(dataset_mlp, val_indices)

# Split Transformer Dataset
train_dataset_tf = Subset(dataset_tf, train_indices)
val_dataset_tf = Subset(dataset_tf, val_indices)

print(f"Total valid days: {total_days}")
print(f"Training on {len(train_indices)} days (e.g., Years 1-8)")
print(f"Dropped {window_size} days to mathematically guarantee zero leakage")
print(f"Validating on {len(val_indices)} strictly unseen future days\n")

# 3. Initialize Architectures
n_assets = len(stock_tickers) # 50 stocks
action_dim = len(stock_tickers) + 1 # N stocks + Cash
state_dim = window_size * n_assets # 3000

print("--- INITIALIZING MODELS ---")
denoiser_mlp = MLPDenoiser(
    state_dim=state_dim, 
    action_dim=action_dim, 
    hidden_dim=config["il"]["hidden_dim"],
    t_dim=config["il"]["t_dim"],
    use_layernorm=config["il"]["use_layernorm"]
)

diffusion_mlp = PortDiff(
    denoiser=denoiser_mlp, 
    num_timesteps=config["il"]["num_timesteps"],
    beta_schedule=config["il"]["beta_schedule"]
)

denoiser_tf = TransformerDenoiser(
    n_assets=n_assets, 
    action_dim=action_dim, 
    window_size=window_size,
    hidden_dim=config["il"]["hidden_dim"],
    t_dim=config["il"]["t_dim"],
    n_heads=4,
    n_layers=2
)

diffusion_tf = PortDiff(
    denoiser=denoiser_tf, 
    num_timesteps=config["il"]["num_timesteps"],
    beta_schedule=config["il"]["beta_schedule"]
)

# 4. Train Both Models Back-to-Back
print("\n--- TRAINING MLP ---")
trainer_mlp = PortDiffTrainer(
    model=diffusion_mlp, 
    train_dataset=train_dataset_mlp, 
    val_dataset=val_dataset_mlp, 
    learning_rate=config["il"]["learning_rate"], 
    batch_size=config["il"]["batch_size"]
)
trained_mlp = trainer_mlp.train(epochs=config["il"]["epochs"])

print("\n--- TRAINING TRANSFORMER ---")
trainer_tf = PortDiffTrainer(
    model=diffusion_tf, 
    train_dataset=train_dataset_tf, 
    val_dataset=val_dataset_tf, 
    learning_rate=config["il"]["learning_rate"], 
    batch_size=config["il"]["batch_size"]
)
trained_tf = trainer_tf.train(epochs=config["il"]["epochs"])

Total valid days: 2460
Training on 1968 days (e.g., Years 1-8)
Dropped 60 days to mathematically guarantee zero leakage
Validating on 432 strictly unseen future days

--- INITIALIZING MODELS ---

--- TRAINING MLP ---
Starting training on device: mps
Epoch 001 | Train Loss (MSE): 0.978509 | Val Loss (MSE): 0.851207
Epoch 010 | Train Loss (MSE): 0.201118 | Val Loss (MSE): 0.197280
Epoch 020 | Train Loss (MSE): 0.107676 | Val Loss (MSE): 0.132062
Epoch 030 | Train Loss (MSE): 0.091234 | Val Loss (MSE): 0.106075
Epoch 040 | Train Loss (MSE): 0.068812 | Val Loss (MSE): 0.102507
Epoch 050 | Train Loss (MSE): 0.061758 | Val Loss (MSE): 0.086477
Epoch 060 | Train Loss (MSE): 0.054840 | Val Loss (MSE): 0.075331
Epoch 070 | Train Loss (MSE): 0.050342 | Val Loss (MSE): 0.072950
Epoch 080 | Train Loss (MSE): 0.044338 | Val Loss (MSE): 0.091868
Epoch 090 | Train Loss (MSE): 0.044652 | Val Loss (MSE): 0.095585
Epoch 100 | Train Loss (MSE): 0.036444 | Val Loss (MSE): 0.068742
Epoch 110 | Train Loss (

In [4]:
import torch.nn.functional as F

print("Sampling portfolio weights from MLP (this may take a minute)...")
device = trainer_mlp.device
all_mlp_weights = []
for state, _ in trainer_mlp.val_loader:
    state = state.to(device)
    weights = diffusion_mlp.sample(state)
    all_mlp_weights.append(weights.cpu().numpy())
mlp_weights_arr = np.concatenate(all_mlp_weights, axis=0)

print("Sampling portfolio weights from Transformer (this may take a minute)...")
device = trainer_tf.device
all_tf_weights = []
for state, _ in trainer_tf.val_loader:
    state = state.to(device)
    weights = diffusion_tf.sample(state)
    all_tf_weights.append(weights.cpu().numpy())
tf_weights_arr = np.concatenate(all_tf_weights, axis=0)

# Extract Ground Truths
val_expert_weights = expert_weights_df.iloc[val_indices].to_numpy()
val_returns = sim_stocks.iloc[val_indices].to_numpy()

def evaluate_portfolio(weights, returns):
    # weights: (batch, 51) where the last column is cash
    # returns: (batch, 50)
    stock_weights = weights[:, :-1]
    portfolio_returns = np.sum(stock_weights * returns, axis=1)
    cumulative_return = np.prod(1 + portfolio_returns)
    sharpe_ratio = np.mean(portfolio_returns) / (np.std(portfolio_returns) + 1e-9) * np.sqrt(252)
    return cumulative_return, sharpe_ratio

# Evaluate all 4 strategies
eq_weights = np.ones_like(val_expert_weights) / val_expert_weights.shape[1]
eq_ret, eq_sharpe = evaluate_portfolio(eq_weights, val_returns)
exp_ret, exp_sharpe = evaluate_portfolio(val_expert_weights, val_returns)
mlp_ret, mlp_sharpe = evaluate_portfolio(mlp_weights_arr, val_returns)
tf_ret, tf_sharpe = evaluate_portfolio(tf_weights_arr, val_returns)

print("\n----------------------------------------")
print(f"Equal-Weight Index Return:   {eq_ret:.2f}x | Sharpe: {eq_sharpe:.2f}")
print(f"Markowitz Expert Return:     {exp_ret:.2f}x | Sharpe: {exp_sharpe:.2f}")
print(f"MLP Agent Return:            {mlp_ret:.2f}x | Sharpe: {mlp_sharpe:.2f}")
print(f"Transformer Agent Return:    {tf_ret:.2f}x | Sharpe: {tf_sharpe:.2f}")
print("----------------------------------------")

Sampling portfolio weights from MLP (this may take a minute)...
Sampling portfolio weights from Transformer (this may take a minute)...

----------------------------------------
Equal-Weight Index Return:   1.37x | Sharpe: 0.94
Markowitz Expert Return:     8.23x | Sharpe: 4.25
MLP Agent Return:            2.78x | Sharpe: 2.01
Transformer Agent Return:    1.41x | Sharpe: 0.76
----------------------------------------
